# Test the checkout lift

*Data Analysis in Python · Topic 9.4*

Your product manager's dashboard says B's +10.9% is significant. Test it yourself, first on all users and then on the users outside the in-app browsers. Then put a confidence interval around the comparable result.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that reads the test's two files, the cell that builds the `users` table (topic 9.1.'s steps) and the cell that builds the `fixed_users` table: the users outside the in-app browsers (topic 9.3.'s steps).
2. In the first cell marked ✏️, create `naive_p`: the p-value of a two-sided two-proportion z-test of B's conversion rate against A's, on the `users` table.
3. In the second cell marked ✏️, create, on the `fixed_users` table:
   - `fixed_p`: the p-value of the same test
   - `ci_low` and `ci_high`: the 95% confidence interval of B's conversion rate minus A's, as fractions
   - `significant`: `True` if `fixed_p` is below 0.05
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `naive_p` is about `0.0010`, and `fixed_p` about `0.2665`. `ci_low` is about `-0.0013` and `ci_high` about `0.0048`, and `significant` is `False`.

## Parameters

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
IN_APP_BROWSERS = ["Android Webview", "Safari (in-app)"]

In [ ]:
# The test's two files, with the user IDs as text. Run it, and leave it as it is.
assignments = pd.read_csv(DATA_URL + "ab_test_assignments.csv", dtype={"user_pseudo_id": str}, parse_dates=["assigned_at"])
conversions = pd.read_csv(DATA_URL + "ab_test_conversions.csv", dtype={"user_pseudo_id": str}, parse_dates=["converted_at"])
assignments.shape, conversions.shape

In [ ]:
# One row per user in the test, with converted and revenue (topic 9.1). Run it, and leave it as it is.
users = assignments.merge(conversions[["user_pseudo_id", "revenue"]], on="user_pseudo_id", how="left", validate="one_to_one")
users["converted"] = users["revenue"].notna()
users["revenue"] = users["revenue"].fillna(0)
users.shape

In [ ]:
# The users outside the in-app browsers, which B couldn't log (topic 9.3). Run it, and leave it as it is.
fixed_users = users[~users["browser"].isin(IN_APP_BROWSERS)]
fixed_users.shape

## All users

In [ ]:
# ✏️ Create naive_p


## The comparable users

In [ ]:
# ✏️ Create fixed_p, ci_low, ci_high and significant


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import numpy as np
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.000001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    naive_p = found.get("naive_p")
    if not number(naive_p):
        results.append((False, f"naive_p should be a number: the p-value from proportions_ztest() on the users table. It is {naive_p!r}."))
    elif near(naive_p, 0.000515):
        results.append((False, "naive_p is 0.000515: that's a one-sided test, half the two-sided p-value. Leave out alternative=\"larger\": a checkout change can make things worse too."))
    elif near(naive_p, 0.266533):
        results.append((False, "naive_p is 0.2665: that's the test on the fixed_users table. naive_p is the test on all users, the users table."))
    elif near(naive_p, 0.001093):
        results.append((False, "naive_p is 0.001093: that's chi2_contingency() with Yates' correction. The brief asks for the two-proportion z-test: proportions_ztest()."))
    elif not near(naive_p, 0.001031):
        results.append((False, f"naive_p should be the two-sided z-test's p-value for 1,944 of 38,348 against 1,907 of 41,706: about 0.0010. It is {naive_p:.4g}."))
    else:
        results.append((True, f"On all users, the z-test gives p = {naive_p:.4f}: the dashboard's \"significant\" lift. But those groups aren't comparable."))

    fixed_p, low, high, significant = found.get("fixed_p"), found.get("ci_low"), found.get("ci_high"), found.get("significant")
    if not number(fixed_p):
        results.append((False, f"fixed_p should be a number: the p-value from proportions_ztest() on the fixed_users table. It is {fixed_p!r}."))
    elif near(fixed_p, 0.133266):
        results.append((False, "fixed_p is 0.1333: that's a one-sided test, half the two-sided p-value. A one-sided test chosen after seeing the result isn't a fair test: use the default, two-sided."))
    elif near(fixed_p, 0.866734):
        results.append((False, "fixed_p is 0.8667: a one-sided test in the wrong direction. Use the default two-sided test, with B first."))
    elif near(fixed_p, 0.001031):
        results.append((False, "fixed_p is 0.0010: that's the test on all users. Use the fixed_users table."))
    elif near(fixed_p, 0.273765):
        results.append((False, "fixed_p is 0.2738: that's chi2_contingency() with Yates' correction. It says the same, but the brief asks for the z-test: proportions_ztest()."))
    elif not near(fixed_p, 0.266533):
        results.append((False, f"fixed_p should be the two-sided z-test's p-value for 1,944 of 38,348 against 1,853 of 37,861: about 0.2665. It is {fixed_p:.4g}."))
    elif not number(low) or not number(high):
        results.append((False, "ci_low and ci_high should be numbers: low, high = confint_proportions_2indep(conversions_b, users_b, conversions_a, users_a)."))
    elif near(low, -0.004842, tolerance=0.00001) and near(high, 0.001339, tolerance=0.00001):
        results.append((False, "ci_low and ci_high are A's rate minus B's. Put B first, so the interval is B minus A: positive values mean B converts more."))
    elif near(high, 0.48417, 0.48407, tolerance=0.0001):
        results.append((False, "ci_low and ci_high are in percentage points, -0.13 and 0.48. Keep them as fractions of 1."))
    elif near(low, 0.0020, tolerance=0.0001):
        results.append((False, "ci_low and ci_high are the interval for all users. Use the fixed_users table's counts."))
    elif not (near(low, -0.0013392, -0.0013378, tolerance=0.0000002) and near(high, 0.0048417, 0.0048407, tolerance=0.0000002)):
        results.append((False, f"ci_low and ci_high should be the 95% interval of B's rate minus A's on the fixed_users table: about -0.0013 and 0.0048. They are {low:.4f} and {high:.4f}."))
    elif not isinstance(significant, (bool, np.bool_)):
        results.append((False, f"significant should be True or False: fixed_p < 0.05. It is {significant!r}."))
    elif significant:
        results.append((False, "significant is True, but fixed_p is 0.27, above 0.05: significant = fixed_p < 0.05."))
    else:
        results.append((True, f"On comparable users, p = {fixed_p:.2f}: not significant. B's rate minus A's is between {low * 100:+.2f} and {high * 100:+.2f} percentage points, so the redesign could be slightly worse or up to 10% better. This test can't tell."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
counts = users.groupby("variant")["converted"].agg(["sum", "count"])
z, naive_p = proportions_ztest(counts.loc[["B", "A"], "sum"], counts.loc[["B", "A"], "count"])
counts, naive_p
```

```python
fixed_counts = fixed_users.groupby("variant")["converted"].agg(["sum", "count"])
z, fixed_p = proportions_ztest(fixed_counts.loc[["B", "A"], "sum"], fixed_counts.loc[["B", "A"], "count"])
ci_low, ci_high = confint_proportions_2indep(fixed_counts.loc["B", "sum"], fixed_counts.loc["B", "count"], fixed_counts.loc["A", "sum"], fixed_counts.loc["A", "count"])
significant = fixed_p < 0.05
fixed_p, ci_low, ci_high, significant
```

</details>